<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/xGBoostModelreviewed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install --upgrade pandas numpy scikit-learn xgboost

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score,classification_report
from xgboost import XGBClassifier
import pickle
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score

In [1]:
import os

print(os.path.getsize("sparse_dict.pkl"))

735277681


In [2]:
#Load the sparse dataset
import pickle
with open("sparse_dict.pkl", "rb") as f:
  sparse = pickle.load(f)


In [3]:
print(sparse.keys())

dict_keys(['sparse_train', 'sparse_val', 'sparse_test'])


In [4]:
for name, hts in sparse.items():
  print(name, hts.shape)

sparse_train (1271440, 56)
sparse_val (423814, 56)
sparse_test (423814, 56)


In [5]:
cols_to_drop = ["EMR", "SDP", "County","PatientPk","MFL_Code","SiteCode","Latitude","Longitude","Facility_Name", "SubCounty",
                "ClientTestedAs","CoupleDiscordant","VisitDate","NumberOfPartners"]

for key in sparse.keys():
  sparse[key] = sparse[key].drop(columns=cols_to_drop, errors="ignore")


In [6]:
#Check dtypes
for name, df in sparse.items():
    print(f"\n{name}")
    print(df.dtypes.value_counts())



sparse_train
object     41
float64     1
Name: count, dtype: int64

sparse_val
object     41
float64     1
Name: count, dtype: int64

sparse_test
object     41
float64     1
Name: count, dtype: int64


Prepare train, validation and test sets

In [7]:
#define features and target variable
X_train = sparse["sparse_train"].drop(columns=["FinalTestResult"])
y_train = sparse["sparse_train"]["FinalTestResult"]

X_val = sparse["sparse_val"].drop(columns=["FinalTestResult"])
y_val = sparse["sparse_val"]["FinalTestResult"]

X_test = sparse["sparse_test"].drop(columns=["FinalTestResult"])
y_test = sparse["sparse_test"]["FinalTestResult"]

Encode categorical variables

In [10]:
dtype_summary = pd.DataFrame({
    "Variable": X_train.columns,
    "Datatype": X_train.dtypes.astype(str)
})

dtype_summary = dtype_summary.sort_values("Datatype")

dtype_summary.head(100)

,Variable,Datatype
Age,Age,float64
PopulationType,PopulationType,object
KeyPopulation,KeyPopulation,object
PriorityPopulation,PriorityPopulation,object
IsHealthWorker,IsHealthWorker,object
TestedHIVBefore,TestedHIVBefore,object
ResultOfHIV,ResultOfHIV,object
EverHadSex,EverHadSex,object
SexuallyActive,SexuallyActive,object
NewPartner,NewPartner,object


In [11]:
for col in X_train.columns:
    print(col, X_train[col].nunique())


PopulationType 3
KeyPopulation 5
PriorityPopulation 1
IsHealthWorker 3
TestedHIVBefore 2
ResultOfHIV 2
EverHadSex 3
SexuallyActive 5
NewPartner 4
PartnerHIVStatus 5
MultiplePartners 3
AlcoholSex 5
MoneySex 4
CondomBurst 4
UnknownStatusPartner 4
KnownStatusPartner 4
Pregnant 4
BreastfeedingMother 4
ExperiencedViolenceScreening 4
CurrentlyOnPrep 3
TraditionalProcedures 2
MothersStatus 4
ResultOfHIVSelf 2
ScreenedTB 3
TBStatus 4
ReceivedServices 9
ViolenceScreeningType 9
Sex 2
MaritalStatus 7
PatientDisabled 2
EverTestedForHiv 2
MonthsSinceLastTest 5
PriorityPopulationType 5
Age 99
GBVSexual 3
GBVPhysical 3
GBVEmotional 3
ReceivedPREP 2
ReceivedTB 2
ReceivedSTI 2
dayofweek 7


In [12]:
# One-hot encode categorical features into sparse matrices
from scipy import sparse as scipy_sparse
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns

encoder = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), categorical_cols)
    ],
    remainder="passthrough",
    sparse_threshold=1.0,
)

X_train_enc = scipy_sparse.csr_matrix(encoder.fit_transform(X_train))
X_val_enc = scipy_sparse.csr_matrix(encoder.transform(X_val))
X_test_enc = scipy_sparse.csr_matrix(encoder.transform(X_test))
encoded_feature_names = encoder.get_feature_names_out()

In [13]:
# Verify that all splits have the same encoded features
print(X_train_enc.shape[1] == X_val_enc.shape[1])
print(X_train_enc.shape[1] == X_test_enc.shape[1])

True
True


In [14]:
# Preview a small dense slice without expanding the full sparse matrix
pd.DataFrame(
    X_train_enc[:5, :10].toarray(),
    columns=encoded_feature_names[:10],
)

,categorical__PopulationType_GP,categorical__PopulationType_KP,categorical__PopulationType_PRIORITY,categorical__KeyPopulation_FSW,categorical__KeyPopulation_MSM,categorical__KeyPopulation_NR,categorical__KeyPopulation_PRISONER,categorical__KeyPopulation_PWID,categorical__KeyPopulation_nan,categorical__PriorityPopulation_NR
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [15]:
print("shape:", X_train_enc.shape)
print("dtype:", X_train_enc.dtype)
print("stored nonzeros:", X_train_enc.nnz)

shape: (1271440, 183)
dtype: float64
stored nonzeros: 52129040


In [16]:
density = X_train_enc.nnz / (X_train_enc.shape[0] * X_train_enc.shape[1])
print(f"Sparse matrix density: {density:.6f}")

Sparse matrix density: 0.224044


Encode target variable

In [17]:
#Check the output, if not in numericals, encode it
print(y_train.value_counts())

FinalTestResult
NEGATIVE    1239327
POSITIVE      32113
Name: count, dtype: int64


In [18]:
# Encode target labels using the training-set mapping
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

In [19]:
#Calculate this if positivity is much fewer than negatives: This is because HIV positivity is usually imbalanced
neg = (y_train_enc == 0).sum()
pos = (y_train_enc == 1).sum()

scale_pos_weight = neg / pos

print(scale_pos_weight)

38.592688319372215


Updated code

In [20]:
from itertools import product
from xgboost import XGBClassifier
import pandas as pd
import time

# Calculate imbalance ratio
pos_rate = (y_train == "POSITIVE").mean()
scale_weight = (1 - pos_rate) / pos_rate

print(f"Positivity rate: {pos_rate:.4f}")
print(f"Suggested scale_pos_weight: {scale_weight:.1f}")

# Expanded search space
max_depths = [3, 5, 7, 9]
learning_rates = [0.01, 0.03, 0.05, 0.1]
scale_pos_weights = [1, 10, 20, 30, int(scale_weight)]
subsamples = [0.7, 0.8, 1.0]
colsample_bytrees = [0.6, 0.8, 1.0]
min_child_weights = [1, 5, 10]

results = []

best_model = None
best_score = float("-inf")
best_params = None

configurations = list(product(
    max_depths,
    learning_rates,
    scale_pos_weights,
    subsamples,
    colsample_bytrees,
    min_child_weights
))

print(f"Total configurations: {len(configurations)}")

for i, (
    max_depth,
    learning_rate,
    scale_pos_weight,
    subsample,
    colsample_bytree,
    min_child_weight
) in enumerate(configurations, start=1):

    print(f"Running {i}/{len(configurations)}")

    model = XGBClassifier(
        n_estimators=2000,
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=scale_pos_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        reg_alpha=0.5,
        reg_lambda=2,
        gamma=0.1,
        eval_metric="aucpr",
        early_stopping_rounds=100,
        random_state=2231,
        n_jobs=-1
    )

    model.fit(
        X_train_enc,
        y_train_enc,
        eval_set=[(X_val_enc, y_val_enc)],
        verbose=False
    )

    score = model.best_score

    results.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "scale_pos_weight": scale_pos_weight,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "validation_aucpr": score,
        "best_iteration": model.best_iteration
    })

    if score > best_score:
        best_score = score
        best_model = model
        best_params = results[-1]

results_df = pd.DataFrame(results).sort_values(
    "validation_aucpr",
    ascending=False
)

print("\nBest Parameters")
print(best_params)

print("\nBest Validation AUPRC")
print(best_score)

results_df.to_csv(
    "xgb_hyperparameter_results_v2.csv",
    index=False
)

Positivity rate: 0.0253
Suggested scale_pos_weight: 38.6
Total configurations: 2160
Running 1/2160
Running 2/2160
Running 3/2160


KeyboardInterrupt: 

Train XGBoost

In [ ]:
# CHANGED: Define the hyperparameter grid and retain the best model/results.
from itertools import product
import time
import pandas as pd

max_depths = [5, 7, 9]
learning_rates = [0.05, 0.1]
scale_pos_weights = [1, 10]
subsamples = [0.8, 1.0]

# CHANGED: Store every validation score instead of only overwriting xgb_model.
results = []
best_model = None
best_score = float("-inf")
best_params = None

# CHANGED: Track progress and estimate the remaining runtime.
configurations = list(product(
    max_depths,
    learning_rates,
    scale_pos_weights,
    subsamples
))
total_runs = len(configurations)
search_start = time.perf_counter()

for run_number, (max_depth, learning_rate, scale_pos_weight, subsample) in enumerate(
    configurations, start=1
):
    print(
        f"Starting {run_number}/{total_runs} ({run_number / total_runs:.1%}): "
        f"depth={max_depth}, lr={learning_rate}, spw={scale_pos_weight}, "
        f"subsample={subsample}",
        flush=True,
    )

    xgb_model = XGBClassifier(
        n_estimators=1000,
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=scale_pos_weight,
        subsample=subsample,
        random_state=2231,
        eval_metric="aucpr",
        early_stopping_rounds=50,
        n_jobs=-1
    )

    xgb_model.fit(
        X_train_enc,
        y_train_enc,
        eval_set=[(X_val_enc, y_val_enc)],
        verbose=False
    )

    score = xgb_model.best_score
    params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "scale_pos_weight": scale_pos_weight,
        "subsample": subsample,
    }
    results.append({
        **params,
        "validation_aucpr": score,
        "best_iteration": xgb_model.best_iteration,
    })

    # CHANGED: Select the model using validation AUPRC, matching eval_metric.
    if score > best_score:
        best_score = score
        best_model = xgb_model
        best_params = params

    elapsed = time.perf_counter() - search_start
    average_time = elapsed / run_number
    eta_seconds = average_time * (total_runs - run_number)
    print(
        f"Completed {run_number}/{total_runs}; "
        f"AUPRC={score:.5f}; "
        f"elapsed={elapsed / 60:.1f} min; "
        f"ETA={eta_seconds / 60:.1f} min",
        flush=True,
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("validation_aucpr", ascending=False)
    .reset_index(drop=True)
)

results_df.to_csv("xgb_hyperparameter_results.csv", index=False)

expected_runs = len(configurations)
print(f"Completed {len(results)} of {expected_runs} configurations")
print("Best parameters:", best_params)
print("Best validation AUPRC:", best_score)
print("Best iteration:", best_model.best_iteration)
results_df.head(10)

Starting 1/24 (4.2%): depth=5, lr=0.05, spw=1, subsample=0.8
Completed 1/24; AUPRC=0.21757; elapsed=49.2 min; ETA=1132.1 min
Starting 2/24 (8.3%): depth=5, lr=0.05, spw=1, subsample=1.0
Completed 2/24; AUPRC=0.21520; elapsed=96.4 min; ETA=1060.6 min
Starting 3/24 (12.5%): depth=5, lr=0.05, spw=10, subsample=0.8
Completed 3/24; AUPRC=0.21348; elapsed=143.1 min; ETA=1001.7 min
Starting 4/24 (16.7%): depth=5, lr=0.05, spw=10, subsample=1.0
Completed 4/24; AUPRC=0.21230; elapsed=190.6 min; ETA=952.9 min
Starting 5/24 (20.8%): depth=5, lr=0.1, spw=1, subsample=0.8


Adjusted Model

In [ ]:
from itertools import product
import time
import pandas as pd
# ADDED: Import metrics to calculate precision and recall
from sklearn.metrics import recall_score, precision_score

max_depths = [5, 7, 9]
learning_rates = [0.05, 0.1]
# ADDED: Expanded scale_pos_weights to push recall even higher if 10 isn't enough
scale_pos_weights = [1, 5, 10, 25]
subsamples = [0.8, 1.0]

# Target threshold for clinical intervention (e.g., flag if >= 10% risk)
TARGET_THRESHOLD = 0.10

results = []
best_model = None
best_score = float("-inf")
best_params = None

configurations = list(product(max_depths, learning_rates, scale_pos_weights, subsamples))
total_runs = len(configurations)
search_start = time.perf_counter()

for run_number, (max_depth, learning_rate, scale_pos_weight, subsample) in enumerate(configurations, start=1):
    print(f"Starting {run_number}/{total_runs} ({run_number / total_runs:.1%}): depth={max_depth}, lr={learning_rate}, spw={scale_pos_weight}, subsample={subsample}", flush=True)

    xgb_model = XGBClassifier(
        n_estimators=1000,
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=scale_pos_weight,
        subsample=subsample,
        random_state=2231,
        eval_metric="aucpr",  # Keep AUCPR as the training driver to ensure overall model quality
        early_stopping_rounds=50,
        n_jobs=-1
    )

    xgb_model.fit(
        X_train_enc,
        y_train_enc,
        eval_set=[(X_val_enc, y_val_enc)],
        verbose=False
    )

    aucpr_score = xgb_model.best_score

    # ADDED: Predict probabilities on the validation set
    y_val_probs = xgb_model.predict_proba(X_val_enc)[:, 1]

    # ADDED: Apply the custom decision threshold
    y_val_pred_custom = (y_val_probs >= TARGET_THRESHOLD).astype(int)

    # ADDED: Calculate Recall and Precision at this threshold
    val_recall = recall_score(y_val_enc, y_val_pred_custom, zero_division=0)
    val_precision = precision_score(y_val_enc, y_val_pred_custom, zero_division=0)

    params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "scale_pos_weight": scale_pos_weight,
        "subsample": subsample,
    }

    results.append({
        **params,
        "validation_aucpr": aucpr_score,
        # ADDED: Save recall and precision metrics to the dataframe
        f"val_recall_at_{TARGET_THRESHOLD}": val_recall,
        f"val_precision_at_{TARGET_THRESHOLD}": val_precision,
        "best_iteration": xgb_model.best_iteration,
    })

    # CHANGED: Keep saving the best model based on overall AUCPR quality
    if aucpr_score > best_score:
        best_score = aucpr_score
        best_model = xgb_model
        best_params = params

    elapsed = time.perf_counter() - search_start
    average_time = elapsed / run_number
    eta_seconds = average_time * (total_runs - run_number)
    print(f"Completed {run_number}/{total_runs}; AUPRC={aucpr_score:.5f}; Recall={val_recall:.2%}; elapsed={elapsed / 60:.1f} min; ETA={eta_seconds / 60:.1f} min", flush=True)

results_df = pd.DataFrame(results)

# CHANGED: Sort by Recall first, then by AUCPR to find high-coverage models
recall_col = f"val_recall_at_{TARGET_THRESHOLD}"
results_df = results_df.sort_values(by=[recall_col, "validation_aucpr"], ascending=[False, False]).reset_index(drop=True)

results_df.to_csv("xgb_hyperparameter_results.csv", index=False)

expected_runs = len(configurations)
print(f"Completed {len(results)} of {expected_runs} configurations")
results_df.head(10)


Starting 1/48 (2.1%): depth=5, lr=0.05, spw=1, subsample=0.8


KeyboardInterrupt: 

Validate the model

In [ ]:
# CHANGED: Evaluate the selected best_model, not the final loop iteration.
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

val_pred = best_model.predict(X_val_enc)
val_prob = best_model.predict_proba(X_val_enc)[:, 1]

In [ ]:
#Check Accuracy
print("Validation Accuracy:",
      accuracy_score(y_val_enc, val_pred))

Validation Accuracy: 0.9752344188724299


In [ ]:
#Check AUC
print("Validation AUC:",
      roc_auc_score(y_val_enc, val_prob))

Validation AUC: 0.8480975356766434


In [ ]:
print(classification_report(y_val_enc, val_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    413110
           1       0.62      0.05      0.09     10704

    accuracy                           0.98    423814
   macro avg       0.80      0.52      0.54    423814
weighted avg       0.97      0.98      0.96    423814



In [ ]:
#Check ROC-AUC
from sklearn.metrics import roc_auc_score

val_prob = best_model.predict_proba(X_val_enc)[:, 1]

roc_auc_score(y_val_enc, val_prob)

0.8480975356766434

In [ ]:
#Check PR-AUC
from sklearn.metrics import average_precision_score

average_precision_score(y_val_enc, val_prob)

0.21886760549483883

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val_enc, val_pred)
print(cm)

[[412777    333]
 [ 10163    541]]


Final test evaluation (run once after model selection)

In [ ]:
# CHANGED: Evaluate on the untouched test set only after selecting hyperparameters.
test_pred = best_model.predict(X_test_enc)
test_prob = best_model.predict_proba(X_test_enc)[:, 1]

print("Test ROC-AUC:", roc_auc_score(y_test_enc, test_prob))
print("Test PR-AUC:", average_precision_score(y_test_enc, test_prob))
print(classification_report(y_test_enc, test_pred))
print("Test confusion matrix:\n", confusion_matrix(y_test_enc, test_pred))

Test ROC-AUC: 0.8427544911254848
Test PR-AUC: 0.21428309824077052
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    413110
           1       0.60      0.05      0.09     10704

    accuracy                           0.98    423814
   macro avg       0.79      0.52      0.54    423814
weighted avg       0.97      0.98      0.96    423814

Test confusion matrix:
 [[412772    338]
 [ 10205    499]]
